# Notebook 02: Baseline Prompting Strategies (Zero-Shot, One-Shot, Static Few-Shot)
### Evaluating Static Demonstration Scales on the Validation Benchmark

This notebook benchmarks three baseline prompting paradigms on a controlled 500-sample validation split:
1. **Zero-Shot**: Task instructions + full 77 intent catalogue + target query (no demonstrations).
2. **One-Shot**: Single canonical demonstration (`card_arrival`) + target query.
3. **Static Few-Shot**: Five fixed demonstrations strictly sourced from the development pool.
4. Latency profiling, token consumption, and financial cost estimation via Groq.


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Initialize Master Pipeline and Validation Subset (500 Samples)


In [ ]:
from src.data.loader import BankingDataLoader
from src.pipeline import BankingIntentPipeline
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.cost_analysis import CostAnalyzer
import pandas as pd

loader = BankingDataLoader()
train_pool, val_df, test_df = loader.load_processed_splits()
eval_val = val_df.head(500).copy()
print(f"Validation evaluation subset: {len(eval_val)} samples across {eval_val['category'].nunique()} intents.")

pipeline = BankingIntentPipeline()


### 2. Experiment 1: Zero-Shot Prompting


In [ ]:
zero_shot_df = pipeline.evaluate_dataset(eval_val, strategy="zero_shot")
zero_shot_df.to_csv("results/tables/zero_shot_results.csv", index=False)

zero_metrics = ClassificationMetrics.compute_all_metrics(
    zero_shot_df["true_intent"].tolist(),
    zero_shot_df["predicted_intent"].tolist()
)
print("Zero-Shot Metrics:", zero_metrics)


### 3. Experiment 2: One-Shot Prompting


In [ ]:
one_shot_df = pipeline.evaluate_dataset(eval_val, strategy="one_shot")
one_shot_df.to_csv("results/tables/one_shot_results.csv", index=False)

one_metrics = ClassificationMetrics.compute_all_metrics(
    one_shot_df["true_intent"].tolist(),
    one_shot_df["predicted_intent"].tolist()
)
print("One-Shot Metrics:", one_metrics)


### 4. Experiment 3: Static Few-Shot Prompting (5 Demonstrations)


In [ ]:
few_shot_df = pipeline.evaluate_dataset(eval_val, strategy="few_shot")
few_shot_df.to_csv("results/tables/few_shot_results.csv", index=False)

few_metrics = ClassificationMetrics.compute_all_metrics(
    few_shot_df["true_intent"].tolist(),
    few_shot_df["predicted_intent"].tolist()
)
print("Few-Shot Metrics:", few_metrics)


### 5. Comparative Analysis: Accuracy, Macro-F1, Latency, and Cost


In [ ]:
cost_analyzer = CostAnalyzer(input_price_per_million=0.20, output_price_per_million=0.20)

def extract_summary(df, name):
    m = ClassificationMetrics.compute_all_metrics(df["true_intent"].tolist(), df["predicted_intent"].tolist())
    c = cost_analyzer.summarize_benchmark_run(df["latency_ms"].tolist(), df["input_tokens"].tolist(), df["output_tokens"].tolist())
    return {
        "Strategy": name,
        "Accuracy": round(m["accuracy"], 4),
        "Macro-F1": round(m["macro_f1"], 4),
        "Weighted-F1": round(m["weighted_f1"], 4),
        "Avg Tokens": round(c["avg_total_tokens"], 1),
        "P95 Latency (ms)": round(c["latency_p95_ms"], 1),
        "Cost / 1K Queries ($)": round(c["cost_per_1k_queries_usd"], 4),
    }

comp_df = pd.DataFrame([
    extract_summary(zero_shot_df, "Zero-shot"),
    extract_summary(one_shot_df, "One-shot"),
    extract_summary(few_shot_df, "Static Few-shot (5)")
])
display(comp_df)
